In [1]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import pickle
import pandas as pd
import math
from pathlib import Path

import matplotlib.pyplot as plt

from matplotlib.colors import PowerNorm
from matplotlib.colors import LogNorm
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap

from matrix_processing_helpers import sparsify_global_percentile
from plot_helpers import plot_rollout_graph, plot_feature_ranking_comparison

In [2]:
with open("data_processed/dataframes/input_cols_pruned.pkl", "rb") as f:
    INPUT_COLS_PRUNED = pickle.load(f)

feature_names = list(INPUT_COLS_PRUNED)

### Attention

For each given number of layers $L$ we have attention matrices $\{A^{(\ell)}\}_{\ell=1}^L$.
We use them to compute the rollout matrix 
$$ R = \hat A^{(L)} \hat A^{(L-1)} \cdot \hat A^{(1)}\in\mathbb{R}^{d\times d}$$

where $\hat A^{(\ell)}$ is the row-normalization of $A^{(\ell)}$. We then compute the **attention-based feature importance scores** summing over the columns of $R$:
$$ s_j = \sum_{i=1}^d R_{ij}, \quad j\in\{1,\ldots,d\}$$


In [3]:
NUM_LAYERS = [3, 5, 7, 9, 11, 13, 15, 17, 20] 
SEED = 1234

ROLLDIR = Path(f"Results/rollout_global/seed_{SEED}")

for layer in NUM_LAYERS:
    SCOREDIR = Path(f"Results/attention_scores/seed_{SEED}/{layer}layers")
    SCOREDIR.mkdir(parents=True, exist_ok=True)
    rollouts = np.load(ROLLDIR / f"{layer}layers/rollout{layer}layers.npz")

    R = rollouts['rollout']
    
    importance = R.sum(axis=0)    
    
    df = pd.DataFrame({'feature': feature_names, 
                       'score': importance,      
                       'original_position': range(len(feature_names))
                      })

    df_sorted = df.sort_values(by='score', ascending=False).reset_index(drop=True)    
    df_sorted['ranking'] = np.arange(1, len(feature_names)+1)

    df.to_csv(SCOREDIR / f"attention_scores{layer}layers_unsorted.csv", 
              index=False, float_format="%.6f")
    
    df_sorted.to_csv(SCOREDIR / f"attention_scores{layer}layers.csv", 
                     index=False, float_format="%.6f")

### SHAP

In [5]:
NUM_LAYERS = [3, 5, 7, 9, 11, 13, 15, 17, 20] 

SHAPDIR = Path(f"Results/shap_values/values/seed_{SEED}")
SHAPDIR_TAB = Path(f"Results/shap_values/tables/seed_{SEED}")

for layer in NUM_LAYERS:
    shap_vals = np.load(SHAPDIR / f"{layer}layers/shap_values_{layer}layers.npz")
    shap_importance = np.mean(np.abs(shap_vals["shap_vals"]), axis=0)

    df_shap = pd.DataFrame({'feature': feature_names,
                            'score': shap_importance,
                            'original_position': range(len(feature_names))
                           })

    df_sorted_shap = df_shap.sort_values(by='score', ascending=False).reset_index(drop=True)
    df_sorted_shap['ranking'] = np.arange(1, len(feature_names)+1)

    df_shap.to_csv(SHAPDIR_TAB / f"{layer}layers/shap_scores{layer}layers_unsorted.csv", 
                   index=False, float_format="%.6f")

    df_sorted_shap.to_csv(SHAPDIR_TAB / f"{layer}layers/shap_scores{layer}layers.csv", 
                          index=False, float_format="%.6f")